### README first

Have a look at the README.md of this subfolder for instructions on how to download and use local Ollama models.

In [ ]:
from ollama import chat
from ollama import ChatResponse

### Models compatible with tools

Some models appear to be compatible with LangChain tools, but they do not seem to be able to call them successfully. From the models below, qwen3 models (even the 4b one) appear to be able to call the tools.

In [ ]:
# model_name = 'qwen3-vl:8b'
model_name = 'qwen3-vl:4b'
# model_name = 'llama3.2:3b'
# model_name = 'llama3.2:1b'

### Running a model via Ollama

Ollama can rand the model via its interface.

In [2]:
response: ChatResponse = chat(model=model_name, messages=[
  {
    'role': 'user',
    'content': 'Why is the sky blue?',
  },
])
print(response['message']['content'])
# or access fields directly from the response object
print(response.message.content)

The sky appears blue due to a phenomenon called **Rayleigh scattering**, which is how light interacts with the molecules and small particles in Earth's atmosphere. Here's a step-by-step explanation:

---

### 1. **Sunlight is white light (all colors)**  
   - The sun emits light consisting of a **full spectrum** of visible colors (violet, indigo, blue, green, yellow, orange, red).  
   - When this light travels through space, it reaches Earth as **white light** (a mixture of all wavelengths).

---

### 2. **Atmospheric molecules scatter light**  
   - As sunlight enters Earth's atmosphere, it collides with **gaseous molecules** (mostly nitrogen and oxygen, but also other particles).  
   - These molecules are **much smaller than the wavelength of visible light** (e.g., a nitrogen molecule is about 0.3 nm, while blue light is 450 nm).  
   - **Rayleigh scattering** occurs when light is scattered by particles much smaller than its wavelength. This scattering is **stronger for shorter wav

### LangChain Tools

Let's integrate a web search tool with an Ollama model. First, let's check out the duckduckgo search tool via python. In this tool we can submit queries and get snippets (summaries + titles) fromt some top results (like the snippets we get from results when we search in Google).

In [3]:
# Integrating DuckDuckGo as a tool
from langchain_community.tools import DuckDuckGoSearchRun
ddg_search = DuckDuckGoSearchRun()
# ddg_search.invoke('Why do parrots have colorful feathers?')
ddg_search.run('Who is the president of Venezuela?')

"Nicolas Maduro served as president of Venezuela for more than 10 years before his ouster over the weekend. Who is Nicolás Maduro? Nicolás Maduro, 63, is the president of Venezuela and has served as the oil-rich country's leader since 2013. He was born in the country's capital of Caracas. Venezuelan President Nicolas Maduro was sworn in for a third term in January 2025 following a 2024 election that was widely condemned by international observers and ... Venezuelan President Hugo Chavez gives his first press conference after winning national elections on October 9, 2012, in Caracas, Venezuela . Bus driver turned president led Venezuela with a heavy hand After succeeding Hugo Chavez in 2013, Maduro persecuted political opponents and repeatedly staged sham elections, while his supporters ..."

### Initialize Ollama model via LangChain

We can initialize an Ollama model for chat via LangChain.

In [4]:
from langchain_ollama import ChatOllama
model = ChatOllama(
    model=model_name,
    validate_model_on_init=True,
    temperature=0,
)

### LangChain invoking

We can also invoke the model via LangChain.

In [5]:
response = model.invoke("Who is the president of Venezuela?")
print(response)

content='As of my latest update (July 2024), **Nicolás Maduro** is the **de facto president of Venezuela**, having assumed office in 2013 following the death of Hugo Chávez and the disputed 2013 election. However, the situation is highly complex due to ongoing political instability and international disputes:\n\n1. **Official Status**:  \n   - Maduro was elected in the **2013 presidential election** (which the opposition and many international observers called "unfree" due to alleged fraud).  \n   - He has been re-elected in **2018** (with the UN and most countries recognizing it as legitimate) and **2024** (a disputed election where the opposition rejected the results).  \n   - The **UN, the United States, and most Western nations** recognize Maduro as Venezuela\'s president, though they criticize his legitimacy and human rights record.  \n\n2. **Key Context**:  \n   - **Opposition Claims**: The opposition (led by Juan Guaidó) has declared Maduro illegitimate since 2019, claiming the 

### Tool preparation

We can prepare the search tool above for LangChain and bind the tool to the model.

In [6]:
from langchain.tools import tool

@tool
def search(query: str) -> str:
    """Search for information on the internet using a search tool."""
    print('TOOL CALLED: ', query)
    return ddg_search.run(query)

In [7]:
model_with_tool = model.bind_tools([search])

In [8]:
from langchain_core.messages import HumanMessage, ToolMessage

### Tool usage and messages

If we are not using agents, we need to orchestrate the messaging so that the model calls the tool properly and integrates the results to its response.

In [9]:
messages = [
    HumanMessage(
        content="You must use the search tool to answer this question: "
                "Who is the president of Venezuela?"
    )
]

# First turn
ai_msg = model_with_tool.invoke(messages)
messages.append(ai_msg)

# Tool execution
if ai_msg.tool_calls:
    for tool_call in ai_msg.tool_calls:
        print(tool_call)
        # Corrected: Access 'args' as a dictionary key and then 'query' within it
        tool_result = search.invoke(tool_call['args']['query'])
        messages.append(
            ToolMessage(
                content=str(tool_result),
                tool_call_id=tool_call['id'] # Access 'id' as a dictionary key as well
            )
        )

    # New human turn (required!)
    messages.append(
        HumanMessage(
            content="Using the tool results above, answer the question."
        )
    )

    final_response = model_with_tool.invoke(messages)
    print(final_response.content)
else:
    print("Model did not call tool:")
    print(ai_msg.content)

{'name': 'search', 'args': {'query': 'current president of Venezuela'}, 'id': '0b31f788-ac9b-4fed-8434-f4d923e8a902', 'type': 'tool_call'}
TOOL CALLED:  current president of Venezuela
Based on the most recent information provided in the search results, **Delcy Rodríguez** is the **acting president of Venezuela**. This follows the U.S. capture of Nicolás Maduro and his wife Cilia Flores on January 3, 2026, which led to Rodríguez assuming the role of interim leader. The search results confirm her position as the current acting president, with Maduro's removal from office marking a significant political shift.
